# Netflix RecSys — Model Development & Comparison
SVD vs User-Based CF | RMSE + MAP@10

In [ ]:
# !pip install scikit-surprise

In [ ]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD, KNNBasic
from surprise.model_selection import train_test_split
from surprise import accuracy
from collections import defaultdict

print('Libraries loaded ✅')

## 1. Load Data

In [ ]:
df     = pd.read_csv('data/netflix_clean.csv', parse_dates=['date'])
movies = pd.read_csv('data/movies_clean.csv')

print(f'Ratings : {len(df):,}')
print(f'Users   : {df.user_id.nunique():,}')
print(f'Movies  : {df.movie_id.nunique():,}')
df.head()

## 2. Train / Test Split (time-based)
Train on ratings before 2005-10-01, test on after. Keeps temporal integrity.

In [ ]:
SPLIT_DATE = '2005-10-01'

train_df = df[df['date'] <  SPLIT_DATE].copy()
test_df  = df[df['date'] >= SPLIT_DATE].copy()

# Keep only test users/movies that exist in train
train_users  = set(train_df['user_id'])
train_movies = set(train_df['movie_id'])
test_df = test_df[
    test_df['user_id'].isin(train_users) &
    test_df['movie_id'].isin(train_movies)
].copy()

print(f'Train : {len(train_df):,}  ({train_df.date.min().date()} → {train_df.date.max().date()})')
print(f'Test  : {len(test_df):,}  ({test_df.date.min().date()} → {test_df.date.max().date()})')

## 3. Build Surprise Datasets

In [ ]:
reader = Reader(rating_scale=(1, 5))

train_data = Dataset.load_from_df(
    train_df[['user_id', 'movie_id', 'rating']], reader
).build_full_trainset()

# Testset from test_df
testset = [
    (row.user_id, row.movie_id, row.rating)
    for row in test_df[['user_id', 'movie_id', 'rating']].itertuples()
]

print(f'Trainset size : {train_data.n_ratings:,}')
print(f'Testset size  : {len(testset):,}')

## 4. Model 1 — SVD (Matrix Factorization)

In [ ]:
print('Training SVD...')
svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(train_data)

svd_preds = svd.test(testset)
svd_rmse  = accuracy.rmse(svd_preds, verbose=False)
svd_mae   = accuracy.mae(svd_preds,  verbose=False)

print(f'SVD  RMSE : {svd_rmse:.4f}')
print(f'SVD  MAE  : {svd_mae:.4f}')

## 5. Model 2 — User-Based CF (KNN)

In [ ]:
print('Training User-Based CF...')
ubcf = KNNBasic(
    k=40, min_k=5,
    sim_options={'name': 'cosine', 'user_based': True},
    verbose=False
)
ubcf.fit(train_data)

ubcf_preds = ubcf.test(testset)
ubcf_rmse  = accuracy.rmse(ubcf_preds, verbose=False)
ubcf_mae   = accuracy.mae(ubcf_preds,  verbose=False)

print(f'UBCF RMSE : {ubcf_rmse:.4f}')
print(f'UBCF MAE  : {ubcf_mae:.4f}')

## 6. MAP@10
Relevant = actual rating ≥ 3.5 (as specified in problem statement).
For each test user, score all their test movies with the model, rank, take top-10.

In [ ]:
RELEVANCE_THRESHOLD = 3.5

def compute_map_at_k(predictions, k=10, threshold=RELEVANCE_THRESHOLD):
    """Compute MAP@K from surprise predictions list."""

    # Group predictions by user
    user_preds = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_preds[uid].append((est, true_r))

    ap_list = []

    for uid, preds in user_preds.items():
        # Only score users that have at least one relevant item
        relevant_total = sum(1 for _, true_r in preds if true_r >= threshold)
        if relevant_total == 0:
            continue

        # Sort by estimated rating descending → top-K
        preds_sorted = sorted(preds, key=lambda x: x[0], reverse=True)[:k]

        hits = 0
        precision_sum = 0.0

        for rank, (est, true_r) in enumerate(preds_sorted, start=1):
            if true_r >= threshold:
                hits += 1
                precision_sum += hits / rank

        ap = precision_sum / min(relevant_total, k)
        ap_list.append(ap)

    return np.mean(ap_list) if ap_list else 0.0


svd_map  = compute_map_at_k(svd_preds,  k=10)
ubcf_map = compute_map_at_k(ubcf_preds, k=10)

print(f'SVD  MAP@10 : {svd_map:.4f}')
print(f'UBCF MAP@10 : {ubcf_map:.4f}')

## 7. Comparison Table

In [ ]:
results = pd.DataFrame({
    'Model'  : ['SVD (MF)', 'User-Based CF'],
    'RMSE'   : [svd_rmse,  ubcf_rmse],
    'MAE'    : [svd_mae,   ubcf_mae],
    'MAP@10' : [svd_map,   ubcf_map],
})

results = results.set_index('Model').round(4)
print(results.to_string())
results

## 8. Top-10 Recommendations — Sample Users

In [ ]:
def get_top_k_recs(model, user_id, train_df, movies_df, k=10):
    """Return top-K unseen movies for a user using the given model."""
    seen = set(train_df[train_df['user_id'] == user_id]['movie_id'])
    all_movies = movies_df['movie_id'].unique()
    unseen = [m for m in all_movies if m not in seen]

    preds = [(m, model.predict(user_id, m).est) for m in unseen]
    preds.sort(key=lambda x: x[1], reverse=True)
    top_k = preds[:k]

    recs = pd.DataFrame(top_k, columns=['movie_id', 'predicted_rating'])
    recs = recs.merge(movies_df[['movie_id', 'title', 'year']], on='movie_id')
    return recs[['title', 'year', 'predicted_rating']].round(3)


# Pick 3 sample users from test set
sample_users = test_df['user_id'].value_counts().head(3).index.tolist()

for uid in sample_users:
    print(f'\n--- User {uid} | SVD Top-10 ---')
    print(get_top_k_recs(svd, uid, train_df, movies).to_string(index=False))

## 9. Success & Failure Cases

In [ ]:
# Predictions as DataFrame for easy analysis
svd_pred_df = pd.DataFrame([
    {'user_id': uid, 'movie_id': iid, 'true_r': tr, 'est_r': est}
    for uid, iid, tr, est, _ in svd_preds
])
svd_pred_df['error'] = abs(svd_pred_df['true_r'] - svd_pred_df['est_r'])
svd_pred_df = svd_pred_df.merge(movies[['movie_id', 'title']], on='movie_id', how='left')

print('=== Success Cases (error < 0.2) ===')
print(svd_pred_df[svd_pred_df['error'] < 0.2]
      [['title','true_r','est_r','error']]
      .sample(5, random_state=1).to_string(index=False))

print('\n=== Failure Cases (error > 2.0) ===')
print(svd_pred_df[svd_pred_df['error'] > 2.0]
      [['title','true_r','est_r','error']]
      .sample(5, random_state=1).to_string(index=False))

## 10. Save Models & Results

In [ ]:
import pickle, os
os.makedirs('models', exist_ok=True)

with open('models/svd.pkl',  'wb') as f: pickle.dump(svd,  f)
with open('models/ubcf.pkl', 'wb') as f: pickle.dump(ubcf, f)

results.to_csv('data/model_results.csv')
print('Models + results saved ✅')